# Test Preprocessing

## Import Modul and Import CSV

In [100]:
import pandas as pd
import importlib
import preprocessing

importlib.reload(preprocessing)

from sklearn.model_selection import train_test_split # Stratified Train-Validation Split 
from preprocessing import (
    # Data Cleaning
    fit_cleaning_params, 
    clean_data,

    # Data Transformation 
    fit_transform_params, 
    transform_data,

    # Feature Selection
    select_features,

    # Dimensionality Reduction
    fit_pca,
    reduce_dimensions)

train = pd.read_csv("../../data/train.csv") # Training Data
test = pd.read_csv("../../data/test.csv") # Testing Data

## 1. Data Cleaning

In [101]:
# Memisahkan fitur dan target
X = train.drop(columns=["id", "is_canceled"])
y = train["is_canceled"]

X_test = test.drop(columns=["id"])

# Membagi data training dan validation
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Menghitung parameter imputasi dari training
params = fit_cleaning_params(X_train)

# Melakukan data cleaning
X_train_clean = clean_data(X_train, params)
X_val_clean = clean_data(X_val, params)
X_test_clean = clean_data(X_test, params)

display(params)

{'medians': num_adults                   2.0
 num_children                 0.0
 days_before_arrival         57.0
 nightly_rate_idr       1647000.0
 special_requests             0.0
 dtype: float64,
 'parking_mode': np.float64(0.0)}

In [102]:
missing = pd.DataFrame({
    "Train Missing (%)": X_train_clean.isna().mean() * 100,
    "Valid Missing (%)": X_val_clean.isna().mean() * 100,
    "Test Missing (%)": X_test_clean.isna().mean() * 100,
})

missing = missing.sort_values(
    "Train Missing (%)",
    ascending=False
)

display(missing.round(2))

,Train Missing (%),Valid Missing (%),Test Missing (%)
num_adults,0.0,0.0,0.0
num_children,0.0,0.0,0.0
weekend_nights,0.0,0.0,0.0
weekday_nights,0.0,0.0,0.0
meal_package,0.0,0.0,0.0
needs_parking,0.0,0.0,0.0
room_category,0.0,0.0,0.0
days_before_arrival,0.0,0.0,0.0
arrival_year,0.0,0.0,0.0
arrival_month,0.0,0.0,0.0


## 2. Data Transformation

In [103]:
# Mempelajari parameter transformasi dari training
transformer = fit_transform_params(X_train_clean)

# Menerapkan transformasi
X_train_transformed = transform_data(X_train_clean, transformer)
X_val_transformed = transform_data(X_val_clean, transformer)
X_test_transformed = transform_data(X_test_clean, transformer)

display(X_train_transformed.head())

,num_adults,num_children,weekend_nights,weekday_nights,days_before_arrival,prior_cancellations,prior_completed_bookings,nightly_rate_idr,special_requests,arrival_year,...,booking_channel_Airline,booking_channel_Complimentary,booking_channel_Corporate,booking_channel_Offline Agent,booking_channel_Online Travel Agent,needs_parking,is_returning_guest,num_children_was_missing,special_requests_was_missing,needs_parking_was_missing
3315,0.290096,-0.257497,1.350452,-1.547794,3.605815,-0.059618,-0.085155,-0.117901,0.499866,0.469489,...,0.0,0.0,0.0,1.0,0.0,0.0,0,0,0,0
9891,0.290096,-0.257497,0.207963,-0.139258,2.633313,-0.059618,-0.085155,-0.376800,-0.773673,0.469489,...,0.0,0.0,0.0,0.0,1.0,0.0,0,0,1,0
11242,-1.664525,-0.257497,-0.934527,-0.139258,0.972942,-0.059618,-0.085155,-0.165779,-0.773673,-2.129974,...,0.0,0.0,0.0,1.0,0.0,0.0,0,0,0,0
41,0.290096,-0.257497,-0.934527,-0.139258,-0.651849,-0.059618,-0.085155,-0.522210,-0.773673,0.469489,...,0.0,0.0,1.0,0.0,0.0,0.0,0,0,0,0
11576,0.290096,-0.257497,-0.934527,0.565009,-0.675568,-0.059618,-0.085155,1.103892,-0.773673,0.469489,...,0.0,0.0,0.0,0.0,1.0,0.0,0,0,1,0


In [104]:
# Memastikan tidak ada missing values
assert not X_train_transformed.isna().any().any()
assert not X_val_transformed.isna().any().any()
assert not X_test_transformed.isna().any().any()

# Memastikan struktur kolom sama
assert X_train_transformed.columns.equals(X_val_transformed.columns)
assert X_train_transformed.columns.equals(X_test_transformed.columns)

# Memastikan jumlah baris tidak berubah
assert len(X_train_transformed) == len(X_train_clean)
assert len(X_val_transformed) == len(X_val_clean)
assert len(X_test_transformed) == len(X_test_clean)

print("Data Transformation Test: PASS")

Data Transformation Test: PASS


## 3. Feature Selection

In [105]:
temporal_cols = [
    "arrival_year",
    "arrival_month",
    "arrival_day"
]

### Baseline

In [106]:
X_train_selected = select_features(
    X_train_transformed,
    include_temporal=True
)

X_val_selected = select_features(
    X_val_transformed,
    include_temporal=True
)

X_test_selected = select_features(
    X_test_transformed,
    include_temporal=True
)

### Without Temporal

In [107]:
X_train_no_temporal = select_features(
    X_train_transformed,
    include_temporal=False
)

X_val_no_temporal = select_features(
    X_val_transformed,
    include_temporal=False
)

X_test_no_temporal = select_features(
    X_test_transformed,
    include_temporal=False
)

### Testing

In [108]:
# Baseline masih memiliki fitur temporal
assert all(col in X_train_selected.columns for col in temporal_cols)

# Skenario kedua tidak memiliki fitur temporal
assert all(col not in X_train_no_temporal.columns for col in temporal_cols)

# Struktur fitur harus konsisten
assert X_train_selected.columns.equals(X_val_selected.columns)
assert X_train_selected.columns.equals(X_test_selected.columns)

assert X_train_no_temporal.columns.equals(X_val_no_temporal.columns)
assert X_train_no_temporal.columns.equals(X_test_no_temporal.columns)

# Jumlah baris tidak berubah
assert len(X_train_selected) == len(X_train_transformed)
assert len(X_train_no_temporal) == len(X_train_transformed)

print("Feature Selection Test: PASS")

Feature Selection Test: PASS


# 4. Dimensionality Reduction

## With Temporal

In [109]:
pca_temporal = fit_pca(X_train_selected)

X_train_pca_temporal = reduce_dimensions(X_train_selected, pca_temporal)
X_val_pca_temporal = reduce_dimensions(X_val_selected, pca_temporal)
X_test_pca_temporal = reduce_dimensions(X_test_selected, pca_temporal)

print("Sebelum PCA:", X_train_selected.shape)
print("Sesudah PCA:", X_train_pca_temporal.shape)
print("Explained Variance:", pca_temporal.explained_variance_ratio_.sum())

Sebelum PCA: (21898, 35)
Sesudah PCA: (21898, 15)
Explained Variance: 0.9573072062808238


## Without Temporal

In [110]:
pca_no_temporal = fit_pca(X_train_no_temporal)

X_train_pca_no_temporal = reduce_dimensions(X_train_no_temporal, pca_no_temporal)
X_val_pca_no_temporal = reduce_dimensions(X_val_no_temporal, pca_no_temporal)
X_test_pca_no_temporal = reduce_dimensions(X_test_no_temporal, pca_no_temporal)

print("Sebelum PCA:", X_train_no_temporal.shape)
print("Sesudah PCA:", X_train_pca_no_temporal.shape)
print("Explained Variance:", pca_no_temporal.explained_variance_ratio_.sum())

Sebelum PCA: (21898, 32)
Sesudah PCA: (21898, 13)
Explained Variance: 0.9539775748040915
